In [4]:


import numpy as np
import pandas as pd

from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, VarianceThreshold
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import cross_val_score

pd.set_option("display.width", 120)
np.random.seed(42)


print("BUILDING A SYNTHETIC TRANSACTIONS DATASET")


n_customers = 200
n_transactions = 2000

customer_ids = np.random.choice(range(1, n_customers + 1), n_transactions)
dates = pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0, 365, n_transactions), unit="D")
amounts = np.random.exponential(50, n_transactions).round(2)
categories = np.random.choice(["electronics", "clothing", "grocery", "books"], n_transactions,
                               p=[0.2, 0.3, 0.4, 0.1])

transactions = pd.DataFrame({
    "customer_id": customer_ids,
    "date": dates,
    "amount": amounts,
    "category": categories
})
print(transactions.head())
print("Shape:", transactions.shape)


print("1) FEATURE CREATION — aggregation, ratio, count, date-derived features")


# --- Aggregation features (customer-level from transaction-level data) ---
customer_features = transactions.groupby("customer_id").agg(
    total_spend=("amount", "sum"),
    avg_spend=("amount", "mean"),
    txn_count=("amount", "count"),
    max_spend=("amount", "max"),
    spend_std=("amount", "std"),
).reset_index()
customer_features["spend_std"] = customer_features["spend_std"].fillna(0)

# --- Ratio feature ---
customer_features["avg_to_max_ratio"] = customer_features["avg_spend"] / customer_features["max_spend"]

# --- Count feature: number of distinct categories a customer buys from ---
category_diversity = transactions.groupby("customer_id")["category"].nunique().rename("category_diversity")
customer_features = customer_features.merge(category_diversity, on="customer_id")

# --- Domain-based feature: days since last purchase (recency) ---
today = transactions["date"].max() + pd.Timedelta(days=1)
recency = transactions.groupby("customer_id")["date"].max().rename("last_purchase_date")
customer_features = customer_features.merge(recency, on="customer_id")
customer_features["days_since_last_purchase"] = (today - customer_features["last_purchase_date"]).dt.days

# --- Domain-based feature: average order value (classic e-commerce KPI) ---
customer_features["avg_order_value"] = customer_features["total_spend"] / customer_features["txn_count"]
customer_features["is_repeat_customer"] = (customer_features["txn_count"] > 1).astype(int)

print(customer_features.head())

# Fabricate a target: "high value customer" (for demonstrating selection/modeling below)
customer_features["high_value"] = (
    customer_features["total_spend"] > customer_features["total_spend"].median()
).astype(int)


print("2) DATE/TIME FEATURES")


transactions["year"] = transactions["date"].dt.year
transactions["month"] = transactions["date"].dt.month
transactions["day_of_week"] = transactions["date"].dt.dayofweek
transactions["is_weekend"] = transactions["day_of_week"].isin([5, 6]).astype(int)
transactions["quarter"] = transactions["date"].dt.quarter

# Cyclical encoding — avoids the false "distance" between Dec (12) and Jan (1)
transactions["month_sin"] = np.sin(2 * np.pi * transactions["month"] / 12)
transactions["month_cos"] = np.cos(2 * np.pi * transactions["month"] / 12)

print(transactions[["date", "month", "day_of_week", "is_weekend", "month_sin", "month_cos"]].head())

# Lag / rolling features (per customer, sorted by date, shifted to avoid leakage)
transactions_sorted = transactions.sort_values(["customer_id", "date"])
transactions_sorted["amount_lag_1"] = transactions_sorted.groupby("customer_id")["amount"].shift(1)
transactions_sorted["amount_rolling_mean_3"] = (
    transactions_sorted.groupby("customer_id")["amount"]
    .transform(lambda s: s.shift(1).rolling(3).mean())
)
print("\nLag/rolling features (sample customer):")
sample_cust = transactions_sorted["customer_id"].iloc[0]
print(transactions_sorted[transactions_sorted["customer_id"] == sample_cust]
      [["date", "amount", "amount_lag_1", "amount_rolling_mean_3"]].head(6))


print("3) CATEGORICAL FEATURES — grouping rare categories, one-hot")


freq = transactions["category"].value_counts(normalize=True)
print("Category frequencies:\n", freq)
rare_categories = freq[freq < 0.15].index.tolist()
transactions["category_grouped"] = transactions["category"].replace(rare_categories, "Other")
print("\nGrouped category counts:\n", transactions["category_grouped"].value_counts())


print("4) INTERACTION FEATURES")


# Domain interaction: how a customer's average spend compares to their category's typical spend
category_avg = transactions.groupby("category")["amount"].transform("mean")
transactions["amount_vs_category_avg"] = transactions["amount"] - category_avg
print(transactions[["category", "amount", "amount_vs_category_avg"]].head())

# Automated polynomial interactions (numeric features from customer_features)
poly_features = customer_features[["avg_spend", "txn_count"]]
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
poly_out = poly.fit_transform(poly_features)
print("\nPolynomial interaction feature names:", poly.get_feature_names_out())
print(pd.DataFrame(poly_out, columns=poly.get_feature_names_out()).head())


print("5) NUMERICAL TRANSFORMATIONS (fixing skew)")

print(f"total_spend skew before log: {customer_features['total_spend'].skew():.3f}")
customer_features["log_total_spend"] = np.log1p(customer_features["total_spend"])
print(f"total_spend skew after log1p: {customer_features['log_total_spend'].skew():.3f}")


print("6) FEATURE SELECTION")


feature_cols = [
    "total_spend", "avg_spend", "txn_count", "max_spend", "spend_std",
    "avg_to_max_ratio", "category_diversity", "days_since_last_purchase",
    "avg_order_value", "is_repeat_customer", "log_total_spend"
]
X = customer_features[feature_cols]
y = customer_features["high_value"]

# --- Filter method: variance threshold ---
vt = VarianceThreshold(threshold=0.01)
vt.fit(X)
print("Low-variance features that would be dropped:",
      [c for c, k in zip(feature_cols, vt.get_support()) if not k])

# --- Filter method: ANOVA F-test / mutual information ---
selector = SelectKBest(score_func=f_classif, k=5)
selector.fit(X, y)
f_scores = pd.Series(selector.scores_, index=feature_cols).sort_values(ascending=False)
print("\nTop features by ANOVA F-test score:\n", f_scores.head(5))

mi_scores = pd.Series(mutual_info_classif(X, y, random_state=42), index=feature_cols).sort_values(ascending=False)
print("\nTop features by mutual information:\n", mi_scores.head(5))

# --- Embedded method: Lasso (L1) coefficients ---
X_scaled = StandardScaler().fit_transform(X)
lasso = LassoCV(cv=5, random_state=42).fit(X_scaled, y)
lasso_importance = pd.Series(np.abs(lasso.coef_), index=feature_cols).sort_values(ascending=False)
print("\nFeature importance via Lasso (L1) coefficients:\n", lasso_importance)

# --- Embedded method: Random Forest importances ---
rf = RandomForestClassifier(n_estimators=300, random_state=42).fit(X, y)
rf_importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nFeature importance via Random Forest:\n", rf_importance)

# --- Validate: does feature selection actually help? ---
top5_features = f_scores.head(5).index.tolist()
score_all = cross_val_score(RandomForestClassifier(n_estimators=200, random_state=42),
                             X, y, cv=5, scoring="roc_auc").mean()
score_top5 = cross_val_score(RandomForestClassifier(n_estimators=200, random_state=42),
                              X[top5_features], y, cv=5, scoring="roc_auc").mean()
print(f"\nCV ROC-AUC using all {len(feature_cols)} features : {score_all:.4f}")
print(f"CV ROC-AUC using top 5 selected features       : {score_top5:.4f}")

print("\nDone.")


BUILDING A SYNTHETIC TRANSACTIONS DATASET
   customer_id       date  amount     category
0          103 2025-04-13    6.40      grocery
1          180 2025-07-13   15.43  electronics
2           93 2025-06-15    0.89      grocery
3           15 2025-12-25   17.36      grocery
4          107 2025-03-31   74.17     clothing
Shape: (2000, 4)
1) FEATURE CREATION — aggregation, ratio, count, date-derived features
   customer_id  total_spend  avg_spend  txn_count  max_spend  spend_std  avg_to_max_ratio  category_diversity  \
0            1       966.34  64.422667         15     245.19  66.719292          0.262746                   4   
1            2       411.93  51.491250          8     202.97  68.501615          0.253689                   4   
2            3       508.59  50.859000         10     148.78  41.920817          0.341840                   3   
3            4       470.15  47.015000         10     136.32  38.690850          0.344887                   4   
4            5       44